# PSF Estimation & Deconvolution Pipeline

This notebook drives the `psfselect` package, which wraps **EPFL's PSF Generator**
(a Java application) through a small Python wrapper. The flow is:

1. **Make the Java wrapper work** and smoke-test it (Part 1).
2. **Pick one `.tif` stack** and derive the optical parameters from its metadata (Part 2).
3. **Compute a PSF with every estimation model** — Born & Wolf, Gibson & Lanni,
   Richards & Wolf (all via the Java JAR) and Variable-RI Gibson & Lanni (Part 3).
4. **Deconvolve** a cropped sub-volume with a **10-iteration Richardson-Lucy** (Parts 4-5).


## Setup (Google Colab)\n\nRun this once. It mounts your Google Drive, then **searches your Drive for\n`psfgenerator.jar`** to locate the `psf` folder automatically — so you don't\nhave to hard-code any path. It also installs the package and a **Java runtime**\n(the EPFL PSF Generator is a Java app; without a JRE the wrapper silently falls\nback to the pure-Python `psfmodels` backend).\n\nIf the search can't find the folder, the error lists your top-level Drive\nfolders so you can set `psf_pkg_dir` by hand.\n

In [6]:
import sys
import os
import glob

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # --- Locate the 'psf' folder automatically -------------------------------
    # We look for 'psfgenerator.jar' anywhere under your Google Drive, so you do
    # NOT have to hard-code the path. If you know the path, set it here instead:
    psf_pkg_dir = None  # e.g. '/content/drive/MyDrive/FYP/code/psf'

    if psf_pkg_dir is None:
        hits = glob.glob('/content/drive/MyDrive/**/psfgenerator.jar', recursive=True)
        if not hits:
            raise FileNotFoundError(
                "Could not find 'psfgenerator.jar' under /content/drive/MyDrive.\n"
                "Open the Files panel (folder icon on the left) and browse to your\n"
                "'psf' folder, then set psf_pkg_dir above to its path.\n"
                "Top-level folders in your Drive:\n  "
                + "\n  ".join(sorted(os.listdir('/content/drive/MyDrive')))
            )
        psf_pkg_dir = os.path.dirname(hits[0])

    os.chdir(psf_pkg_dir)

    # The EPFL PSF Generator is a JAVA application -> install a JRE so the
    # Python wrapper can actually run the JAR (otherwise it falls back to psfmodels).
    !apt-get -qq update && apt-get -qq install -y default-jre
    !pip install -q -r requirements.txt scikit-image
    !pip install -q -e .
else:
    # Run the notebook from the code/psf/ directory locally
    psf_pkg_dir = os.getcwd()
    if psf_pkg_dir not in sys.path:
        sys.path.append(psf_pkg_dir)

print("Working directory:", psf_pkg_dir)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/FYP/code/psf'

In [4]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Point the wrapper at the bundled EPFL PSF Generator JAR
os.environ["PSF_GENERATOR_JAR"] = os.path.join(psf_pkg_dir, "psfgenerator.jar")

from psfselect import MODELS, MODEL_LABELS
from psfselect.parameters import PSFParams, params_from_metadata
from psfselect.backends import render_psf, available_backends
from psfselect.metadata import validate_samples
from psfselect.metadata_leica import extract_metadata
from psfselect.visualize_napari import load_channels
from skimage.restoration import richardson_lucy

print("Known PSF models:", MODELS)

ModuleNotFoundError: No module named 'psfselect'

## 1. Make the Java wrapper work (and test it)

`available_backends()` probes whether the EPFL JAR can be located **and** whether a
Java runtime is on the `PATH`. We then force the `epfl` backend on a single small
PSF to prove the JAR actually computes and returns a volume.


In [5]:
import shutil

print("java on PATH:", shutil.which("java"))
backends = available_backends()
print("Available backends:", backends)

assert backends["epfl"], (
    "EPFL JAR not available.\n"
    " - check PSF_GENERATOR_JAR points to psfgenerator.jar, and\n"
    " - check a Java runtime is installed (in Colab: apt-get install default-jre)."
)

# Smoke-test: force the Java backend on one model and confirm it really ran.
test_params = PSFParams(na=1.0, wavelength_nm=510.0, ni=1.33, ns=1.33,
                        voxel_xy_um=0.1, voxel_z_um=0.3, nx=31, nz=15)
vol, backend_used, cfg = render_psf("born_wolf", test_params, backend="epfl")

print(f"Rendered Born & Wolf via '{backend_used}' backend, shape={vol.shape}")
assert backend_used == "epfl", "Expected the EPFL Java backend to run the JAR!"
print("Java wrapper works.")

java on PATH: /usr/bin/java


NameError: name 'available_backends' is not defined

## 2. Pick a `.tif` and derive parameters from its metadata

We use `16012025_cmlc2_lifeactxnuclear_48hpf.lif - Series005.tif` (a 3-channel,
193-slice, 512x512 zebrafish cardiac stack). Rather than typing optical parameters
by hand, we extract them from the file and let documented defaults fill any gaps.


In [ ]:
# The dataset lives next to the psf folder at  code/data/raw/...  so we build the
# path relative to psf_pkg_dir -> works the same in Colab and locally.
rel = "../data/raw/cmlc2_lifeactXnuclear/48hpf/16012025_cmlc2_lifeactxnuclear_48hpf.lif - Series005.tif"
image_path = os.path.normpath(os.path.join(psf_pkg_dir, rel))
assert os.path.exists(image_path), f"Image not found: {image_path}"
print("Chosen file:", Path(image_path).name)

# Extract metadata -> resolve defaults -> build PSF params.
# nx/nz are kept small so every PSF (and the deconvolution) stays fast here.
meta, info = extract_metadata(image_path)
sample = validate_samples([meta])[0]
params = params_from_metadata(sample, nx=63, nz=31)

print("\nDerived PSF parameters:")
print(f"  NA       = {params.na}")
print(f"  ni / ns  = {params.ni} / {params.ns}")
print(f"  lambda   = {params.wavelength_nm} nm")
print(f"  voxel xy = {params.voxel_xy_um:.4f} um,  z = {params.voxel_z_um:.4f} um")
print(f"  grid     = nx {params.nx}, nz {params.nz}")
if sample.missing_fields:
    print(f"  [defaults applied for: {', '.join(sample.applied_defaults)}]")

## 3. Compute a PSF with every estimation model

`backend="auto"` runs the **EPFL Java JAR** for Born & Wolf, Gibson & Lanni and
Richards & Wolf, and uses the instant `psfmodels` approximation for Variable-RI
Gibson & Lanni (the JAR's VRIGL routine does not reliably terminate). The
`backend=` column below shows which engine produced each PSF.


In [ ]:
psfs = {}
for model in MODELS:
    vol, backend_used, cfg = render_psf(model, params, backend="auto")
    psfs[model] = vol
    print(f"{MODEL_LABELS[model]:52s} -> backend={backend_used:9s} shape={vol.shape}")

# Central z-slice of each PSF
fig, axes = plt.subplots(1, len(psfs), figsize=(4 * len(psfs), 4))
axes = np.atleast_1d(axes)
for ax, (model, vol) in zip(axes, psfs.items()):
    ax.imshow(vol[vol.shape[0] // 2], cmap="magma")
    ax.set_title(model, fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Load the image and crop a sub-volume

`load_channels` normalises any layout to `(C, Z, Y, X)`. We take the first
(fluorescence) channel and a centred crop so the 10 Richardson-Lucy iterations run
quickly in the notebook.


In [ ]:
raw, ch_idx = load_channels(image_path)   # (C, Z, Y, X)
print("Loaded volume (C, Z, Y, X):", raw.shape, " channels:", ch_idx)

CHANNEL = 0                                # fluorescence channel to deconvolve
volume = raw[CHANNEL]                       # (Z, Y, X)


def center_crop(v, z=31, xy=192):
    nz, ny, nx = v.shape
    z, sy, sx = min(z, nz), min(xy, ny), min(xy, nx)
    z0, y0, x0 = (nz - z) // 2, (ny - sy) // 2, (nx - sx) // 2
    return v[z0:z0 + z, y0:y0 + sy, x0:x0 + sx]


image_crop = center_crop(volume, z=31, xy=192)
print("Cropped volume (Z, Y, X):", image_crop.shape)

plt.imshow(image_crop[image_crop.shape[0] // 2], cmap="gray")
plt.title("Raw image (central z-slice, cropped)")
plt.axis("off")
plt.show()

## 5. Richardson-Lucy deconvolution (10 iterations)

We run **10 iterations** of `skimage.restoration.richardson_lucy` with each
model's PSF. The image is normalised to `[0, 1]` and each PSF is given unit
energy (sum = 1) before deconvolution.


In [ ]:
NUM_ITERS = 10

img = image_crop.astype(float)
img /= max(img.max(), 1e-12)

deconvolved = {}
for model, psf in psfs.items():
    print(f"Richardson-Lucy ({NUM_ITERS} iters) with {model} PSF ...")
    # skimage does not normalise the PSF; give it unit energy (sum=1)
    psf_norm = psf / max(float(psf.sum()), 1e-12)
    deconvolved[model] = richardson_lucy(img, psf_norm, num_iter=NUM_ITERS)

print("Deconvolution finished!")

## 6. Results comparison

Central z-slice of the raw crop next to the deconvolution obtained with each PSF model.


In [ ]:
n = len(deconvolved) + 1
fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
zc = img.shape[0] // 2

axes[0].imshow(img[zc], cmap="gray")
axes[0].set_title("Raw (cropped)")
axes[0].axis("off")

for ax, (model, dec) in zip(axes[1:], deconvolved.items()):
    d = dec / max(float(dec.max()), 1e-12)
    ax.imshow(d[zc], cmap="gray")
    ax.set_title(f"RL: {model}", fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.show()